# Phase 6 — RAG Sémantique

**Objectif** : remplacer le RAG lexical (TF-IDF + cosinus) par un RAG sémantique basé sur des
embeddings denses multilingues et un index FAISS.

| Composant | Choix | Justification |
|---|---|---|
| Embeddings | `paraphrase-multilingual-mpnet-base-v2` | Multilingual FR+EN, 768d, state-of-the-art similarité de phrases |
| Vector DB | FAISS IndexFlatIP | Local, rapide, sans serveur ; IP sur vecteurs L2-normalisés = cosinus |
| Base de connaissance | 476 tickets résolus | Même base que Phase 4 (lexical) pour comparaison directe |

> **ADR-004 décidé** : FAISS local (pas ChromaDB/Weaviate) — priorité à la simplicité et
> à la reproductibilité locale sans dépendance externe.

## 1. Imports & configuration

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

ROOT    = Path('..')
FIG_DIR = ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(exist_ok=True)

print('Imports OK')
print(f'RAG base  : {params["data"]["rag_path"]}')
print(f'top_k     : {params["rag"]["top_k"]}')

## 2. Base de connaissance (tickets résolus)

In [ ]:
rag_df = pd.read_csv(ROOT / params['data']['rag_path'], encoding='utf-8-sig')

# Harmoniser le nom Categorie si l'encodage est corrompu
if 'Catégorie' not in rag_df.columns:
    for col in rag_df.columns:
        if 'gorie' in col.lower():
            rag_df = rag_df.rename(columns={col: 'Catégorie'})
            break

print(f'Base RAG : {len(rag_df)} tickets')
print()
print('Distribution par catégorie :')
print(rag_df['Catégorie'].value_counts().to_string())

## 3. Construction de l'index sémantique (FAISS)

In [ ]:
from src.rag.semantic_engine import SemanticRAGEngine

MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
TOP_K      = params['rag']['top_k']
THRESHOLD  = params['rag']['similarity_threshold']

sem_engine = SemanticRAGEngine(
    model_name           = MODEL_NAME,
    top_k                = TOP_K,
    similarity_threshold = THRESHOLD,
)
sem_engine.fit(rag_df)

## 4. Tests qualitatifs

Mêmes 4 cas que le notebook RAG lexical (`04_rag.ipynb`) pour comparer directement.

In [ ]:
from src.data.preprocess import clean_text

cas_tests = [
    {'titre': 'Erreur activation Office 365',
     'desc':  'Microsoft Office affiche message produit non activé',
     'categorie': 'Microsoft'},
    {'titre': 'Problème accès Sage X3',
     'desc':  'Impossible de se connecter à l ERP Sage X3 depuis ce matin',
     'categorie': 'Applicatifs'},
    {'titre': 'Sauvegarde échouée',
     'desc':  'Le système de sauvegarde indique espace disque insuffisant',
     'categorie': 'Backup'},
    {'titre': 'Accès internet lent',
     'desc':  'Connexion internet très lente depuis le bureau au 3ème étage',
     'categorie': 'Réseau'},
]

print('TESTS SEMANTIQUES')
print('=' * 70)
for cas in cas_tests:
    query = clean_text(cas['desc'])
    recs  = sem_engine.recommend(query, category_filter=cas['categorie'])
    print(f"\n{'_'*70}")
    print(f"Ticket    : {cas['titre']}")
    print(f"Catégorie  : {cas['categorie']}")
    print(f"Query     : {query[:70]}...")
    print(f"Résultats ({len(recs)}) :")
    if not recs:
        print('  Aucune solution similaire trouvée')
    else:
        for i, r in enumerate(recs, 1):
            fb = ' [FALLBACK]' if r.get('fallback') else ''
            print(f"  [{i}] Score {r['score']:.4f}{fb}")
            print(f"       Resolution : {r['resolution'][:100]}")

## 5. Évaluation honnête (held-out, sans auto-récupération)

**Piège évité** : si l'index contient le ticket qu'on interroge, la query (titre+description) retrouve sa propre entrée `text_rag` → score artificiel ~1.0. On évalue donc en **held-out** : index sur 80 % des tickets, requêtes sur les 20 % restants (le ticket-requête n'est jamais dans l'index).

Métriques (requête = titre+description seul, **sans filtre catégorie**, scénario ticket entrant) :
- **same-category@1** : le top-1 récupéré appartient-il à la vraie catégorie ? → précision de routage
- **same-Titre@1** : le top-1 partage-t-il le Titre ? → le RAG retrouve le même type de problème
- **score cosinus moyen top-1** : similarité au plus proche *autre* ticket

In [ ]:
from sklearn.model_selection import train_test_split
from src.data.preprocess import clean_text as _clean
import numpy as np

# Split held-out : 80% index / 20% requetes (le ticket-requete n'est jamais indexe)
kb_train, kb_test = train_test_split(rag_df, test_size=0.2, random_state=42)
print(f'Index (train) : {len(kb_train)} tickets | Requetes (test) : {len(kb_test)} tickets')

# Moteur semantique d'EVALUATION (index sur train uniquement)
sem_eval = SemanticRAGEngine(model_name=MODEL_NAME, top_k=TOP_K, similarity_threshold=THRESHOLD)
sem_eval.fit(kb_train)

def eval_engine(engine, kb_test):
    n, cat_ok, titre_ok, scores = 0, 0, 0, []
    per_cat = {}
    for _, row in kb_test.iterrows():
        q = _clean(str(row.get('Titre','')) + ' ' + str(row.get('Description','')))
        if not q.strip():
            continue
        recs = engine.recommend(q, category_filter=None)  # SANS filtre -> vraie precision
        n += 1
        cat = row['Catégorie']
        per_cat.setdefault(cat, {'n':0,'cat_ok':0})
        per_cat[cat]['n'] += 1
        if recs:
            if not recs[0].get('fallback'):
                scores.append(recs[0]['score'])
            if recs[0]['categorie'] == cat:
                cat_ok += 1; per_cat[cat]['cat_ok'] += 1
            if recs[0]['titre'] == str(row['Titre']):
                titre_ok += 1
    return {
        'n': n,
        'same_cat_pct':   round(cat_ok / n * 100, 1) if n else 0,
        'same_titre_pct': round(titre_ok / n * 100, 1) if n else 0,
        'score_moyen':    round(float(np.mean(scores)), 4) if scores else 0.0,
        'per_cat': per_cat,
    }

res_sem = eval_engine(sem_eval, kb_test)

print('\n=== RAG Semantique (held-out, sans filtre categorie) ===')
print(f"Tickets evalues        : {res_sem['n']}")
print(f"same-category@1        : {res_sem['same_cat_pct']}%")
print(f"same-Titre@1           : {res_sem['same_titre_pct']}%")
print(f"score cosinus moyen    : {res_sem['score_moyen']}")

print('\nsame-category@1 par categorie :')
for cat, d in sorted(res_sem['per_cat'].items()):
    pct = round(d['cat_ok']/d['n']*100,1) if d['n'] else 0
    print(f"  {cat:18s} : {pct:>5.1f}%  ({d['cat_ok']}/{d['n']})")

## 6. Comparaison Lexical vs Sémantique

In [ ]:
from src.rag.engine import RAGEngine as LexRAGEngine
import numpy as np

# Moteur lexical d'EVALUATION sur le MEME split train
lex_eval = LexRAGEngine(top_k=TOP_K, similarity_threshold=THRESHOLD)
lex_eval.fit(kb_train)

res_lex = eval_engine(lex_eval, kb_test)

print('=== Comparaison held-out : Lexical vs Semantique (sans filtre categorie) ===\n')
print(f"{'Metrique':22s} | {'Lexical':>10} | {'Semantique':>11} | {'Gain':>8}")
print('-' * 60)
rows = [
    ('same-category@1 (%)', res_lex['same_cat_pct'],   res_sem['same_cat_pct']),
    ('same-Titre@1 (%)',    res_lex['same_titre_pct'], res_sem['same_titre_pct']),
    ('score cosinus top-1', res_lex['score_moyen'],    res_sem['score_moyen']),
]
for name, lex, sem in rows:
    gain = sem - lex
    arrow = '+' if gain > 0 else ('-' if gain < 0 else '=')
    print(f"{name:22s} | {lex:>10.2f} | {sem:>11.2f} | {arrow}{abs(gain):>7.2f}")

# Variables pour la figure et les metadonnees
global_same_cat_sem = res_sem['same_cat_pct']
global_same_cat_lex = res_lex['same_cat_pct']
global_score_sem    = res_sem['score_moyen']
global_score_lex    = res_lex['score_moyen']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['same-category@1 (%)', 'same-Titre@1 (%)', 'score cosinus x100']
lex_vals = [res_lex['same_cat_pct'], res_lex['same_titre_pct'], res_lex['score_moyen']*100]
sem_vals = [res_sem['same_cat_pct'], res_sem['same_titre_pct'], res_sem['score_moyen']*100]

x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, lex_vals, w, label='Lexical (TF-IDF)',    color='#5B9BD5', alpha=0.85)
b2 = ax.bar(x + w/2, sem_vals, w, label='Semantique (FAISS)', color='#ED7D31', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Valeur'); ax.set_ylim(0, 105)
ax.set_title('Phase 6 - Evaluation held-out : Lexical vs Semantique\n(requete = titre+description, sans filtre categorie)', fontsize=11)
ax.legend(); ax.grid(axis='y', alpha=0.3)
for b in list(b1)+list(b2):
    h = b.get_height()
    ax.annotate(f'{h:.1f}', (b.get_x()+b.get_width()/2, h), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
fig.savefig(FIG_DIR / 'rag6_01_comparaison.png', bbox_inches='tight')
plt.show()
print('Figure sauvegardee : notebooks/figures/rag6_01_comparaison.png')

## 7. Sauvegarde du moteur sémantique

In [ ]:
SAVE_DIR = str(ROOT / 'models' / 'semantic_rag')
sem_engine.save(SAVE_DIR)

# Test rechargement
engine_loaded = SemanticRAGEngine.load(SAVE_DIR)
test_recs = engine_loaded.recommend(
    clean_text('microsoft office erreur activation licence'),
    category_filter='Microsoft'
)
print(f'\nTest rechargement : {len(test_recs)} recommandation(s)')
for r in test_recs:
    print(f"  Score {r['score']:.4f} | {r['resolution'][:80]}")

## 8. Résumé Phase 6

In [ ]:
import json as _json

metadata = {
    'phase':                 'Phase 6 - RAG semantique',
    'model_embeddings':      MODEL_NAME,
    'vector_db':             'FAISS IndexFlatIP (cosinus via L2-norm)',
    'embed_dim':             768,
    'n_tickets_indexed':     sem_engine._index.ntotal,
    'top_k':                 TOP_K,
    'similarity_threshold':  THRESHOLD,
    'evaluation':            'held-out 80/20, requete = titre+description, sans filtre categorie',
    'same_category_at1_sem': global_same_cat_sem,
    'same_category_at1_lex': global_same_cat_lex,
    'score_cosinus_top1_sem': global_score_sem,
    'score_cosinus_top1_lex': global_score_lex,
    'index_path':            'models/semantic_rag/',
    'adr004_decision':       'FAISS local - simplicite et reproductibilite sans serveur',
    'prochaine_etape':       'Phase 7 - Evaluation RAGAS + generation avec citations',
}

out_path = ROOT / 'models' / 'semantic_rag_metadata.json'
out_path.write_text(_json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('=' * 60)
print('RESUME - PHASE 6 : RAG Semantique (evaluation held-out)')
print('=' * 60)
for k, v in metadata.items():
    print(f'  {k:25s}: {v}')
print(f'\nMetadonnees : {out_path}')